notebook_05_judge.ipynb
AI-as-Judge — Retrieval Document Quality Evaluation

Input:  retrieval_documents_structured_v2.json

         retrieval_documents_full_v2.json
        product_entities_normalized_v2.json  (ground truth)

 Output: judge_results_structured_v2.json

         judge_results_full_v2.json

         judge_summary_v2.json


Uses gpt-4.1-mini — cheaper model is fine for evaluation tasks

Cost estimate: ~$0.10 for all 228 documents

In [1]:
# ── CELL 1 — Imports & API key ───────────────────────────────────────────────
import os
import json
import pandas as pd
from tqdm import tqdm
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY', '')

if not api_key:
    import getpass
    api_key = getpass.getpass('OpenAI API key: ')

client  = OpenAI(api_key=api_key)
MODEL   = 'gpt-5.4'   # evaluation task — cheaper model is sufficient
print(f'Model: {MODEL}')

Model: gpt-5.4


In [2]:
# ── CELL 2 — Load all files ──────────────────────────────────────────────────
from google.colab import files

print('Upload product_entities_normalized_v2.json')
uploaded = files.upload()
with open('product_entities_normalized_v2.json', encoding='utf-8') as f:
    entities = {e['product_id']: e for e in json.load(f)}

print('Upload retrieval_documents_structured_v2.json')
uploaded = files.upload()
with open('retrieval_documents_structured_v2.json', encoding='utf-8') as f:
    structured_docs = {d['product_id']: d for d in json.load(f)}

print('Upload retrieval_documents_full_v2.json')
uploaded = files.upload()
with open('retrieval_documents_full_v2.json', encoding='utf-8') as f:
    full_docs = {d['product_id']: d for d in json.load(f)}

print(f'\nEntities:    {len(entities)}')
print(f'Structured:  {len(structured_docs)}')
print(f'Full:        {len(full_docs)}')

Upload product_entities_normalized_v2.json


Saving product_entities_normalized_v2.json to product_entities_normalized_v2.json
Upload retrieval_documents_structured_v2.json


Saving retrieval_documents_structured_v2.json to retrieval_documents_structured_v2.json
Upload retrieval_documents_full_v2.json


Saving retrieval_documents_full_v2.json to retrieval_documents_full_v2.json

Entities:    114
Structured:  114
Full:        114


In [3]:
# ── CELL 3 — Judge system prompt ─────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a senior agricultural data quality auditor specialising in Chinese
agri-input products (pesticides, microbial agents, fertilisers, adjuvants).

You evaluate retrieval documents against their structured entity source.
You understand:
- Bacillus-based microbial agents and their target crops and diseases
- Standard agricultural product-type naming conventions
- How dosage ratios, CFU counts, and application methods should be expressed
- The difference between target diseases, target pests, and plant symptoms

Return ONLY valid JSON — no preamble, no markdown, nothing outside the JSON.\
"""

In [4]:
# ── CELL 4 — Structured document judge prompt ────────────────────────────────
STRUCTURED_JUDGE = """\
Evaluate the RETRIEVAL DOCUMENT against the STRUCTURED ENTITY SOURCE (ground truth).

The document is a structured English summary of the product entity data.
It should faithfully represent every field in the source.

## STRUCTURED ENTITY SOURCE (ground truth)
{entity}

## RETRIEVAL DOCUMENT (to evaluate)
{document}

─────────────────────────────────────────────
## SCORING (each 1–10)

1. Completeness: Are all source fields present and fully populated?
   - 10: every field present and complete
   - 7-9: one minor field missing or slightly truncated
   - 4-6: multiple fields missing or a significant field absent
   - 1-3: core identity fields missing

2. Accuracy: Does the document faithfully represent the source without errors?
   - 10: all values match source exactly
   - 7-9: minor formatting differences only
   - 4-6: one factual error or hallucination
   - 1-3: multiple hallucinated or incorrect values

3. Entity Categorisation: Are entities under the correct section headings?
   - 10: every entity correctly placed
   - 7-9: one ambiguous entity misplaced
   - 4-6: multiple miscategorised entities
   - 1-3: fundamental category confusion
   Check: nematodes must be under TARGET PESTS not TARGET DISEASES
   Check: symptoms (yellowing, dwarfing, mosaic) must be under SYMPTOMS ADDRESSED

4. Retrieval Utility: Would this document surface correctly for relevant queries?
   - 10: all searchable terms present — product name, type, crops, diseases
   - 7-9: minor gaps
   - 4-6: key terms missing — common queries would fail
   - 1-3: too sparse or malformed for reliable retrieval

─────────────────────────────────────────────
## ISSUE TAXONOMY
MISSING_FIELD, TRUNCATED_LIST, HALLUCINATION, MISCATEGORISATION,
DUPLICATE, FORMAT_INCONSISTENCY, NUMERIC_MISMATCH

─────────────────────────────────────────────
## OUTPUT — return ONLY this JSON

{{
  "product_id": "",
  "product_name": "",
  "document_type": "structured",
  "scores": {{
    "completeness": 0,
    "accuracy": 0,
    "entity_categorisation": 0,
    "retrieval_utility": 0,
    "overall": 0
  }},
  "issues": [
    {{
      "type": "",
      "section": "",
      "description": "",
      "severity": "low|medium|high"
    }}
  ],
  "strengths": [""],
  "recommendation": ""
}}

Rules:
- overall = round(completeness×0.30 + accuracy×0.30 + entity_categorisation×0.20 + retrieval_utility×0.20)
- Empty fields (None) are acceptable when the product type makes them inapplicable
  (e.g. a herbicide has no target_diseases — do NOT penalise honest nulls)
- issues may be []
- strengths must have at least one entry\
"""

In [14]:
# ── CELL 5 — Full document judge prompt ──────────────────────────────────────
FULL_JUDGE = """\
EEvaluate the RETRIEVAL DOCUMENT against the STRUCTURED ENTITY SOURCE (ground truth).

This document has THREE sections:
  1. PRODUCT SUMMARY — structured English summary
  2. ORIGINAL PRODUCT DESCRIPTION — translated English prose
  3. 中文产品说明 — original Chinese prose

IMPORTANT ARCHITECTURAL NOTE:
The prose sections (ORIGINAL PRODUCT DESCRIPTION and 中文产品说明) are the
original manufacturer label text. They will naturally contain richer detail,
broader efficacy claims, and more context than the structured summary.
This is EXPECTED and CORRECT — do NOT flag it as hallucination or conflict.

Only flag as HALLUCINATION if the prose contradicts the structured source
(e.g. claims a different active ingredient, wrong dosage ratio, wrong product type).
Only flag as CROSS_SECTION_CONFLICT if a specific factual value directly
contradicts between sections (e.g. structured says 5 strains, prose says 3 strains).
Do NOT flag the prose for containing MORE information than the structured section.

Evaluate all three sections.

## STRUCTURED ENTITY SOURCE (ground truth)
{entity}

## RETRIEVAL DOCUMENT (to evaluate)
{document}

─────────────────────────────────────────────
## SCORING (each 1–10)

1. Completeness: Are all source fields in the structured section complete?
   Also: does the prose add context the structured section cannot capture?
   - 10: all fields present; prose adds meaningful extra context
   - 7-9: one minor gap; prose is adequate
   - 4-6: multiple fields missing or prose repeats structured verbatim
   - 1-3: core fields missing from all sections

2. Accuracy: Are all values faithful to the source across all sections?
   - 10: all sections consistent and accurate
   - 7-9: minor wording differences between sections (expected)
   - 4-6: a factual value differs between structured and prose sections
   - 1-3: systematic contradictions between sections

3. Cross-Section Consistency: Do all three sections agree?
   - 10: fully consistent
   - 7-9: minor wording differences only
   - 4-6: a factual value differs (e.g. different ingredients in structured vs prose)
   - 1-3: systematic contradictions
   Check: do disease/pest names in Chinese prose match English structured section?

4. Entity Categorisation: Are entities in the structured section correct?
   Nematodes → TARGET PESTS not TARGET DISEASES
   Symptoms → SYMPTOMS ADDRESSED not TARGET DISEASES

5. Retrieval Utility: Does the full document maximise retrievability?
   - 10: rich coverage — Chinese prose enables Chinese queries, English enables English
   - 7-9: good coverage; minor gaps
   - 4-6: Chinese or English section adds little value
   - 1-3: document too repetitive or malformed

─────────────────────────────────────────────
## ISSUE TAXONOMY
MISSING_FIELD, TRUNCATED_LIST, HALLUCINATION, MISCATEGORISATION,
CROSS_SECTION_CONFLICT, DUPLICATE, FORMAT_INCONSISTENCY, NUMERIC_MISMATCH

─────────────────────────────────────────────
## OUTPUT — return ONLY this JSON

{{
  "product_id": "",
  "product_name": "",
  "document_type": "full",
  "scores": {{
    "completeness": 0,
    "accuracy": 0,
    "cross_section_consistency": 0,
    "entity_categorisation": 0,
    "retrieval_utility": 0,
    "overall": 0
  }},
  "issues": [
    {{
      "type": "",
      "section": "PRODUCT SUMMARY|ORIGINAL PRODUCT DESCRIPTION|中文产品说明|cross-section",
      "description": "",
      "severity": "low|medium|high"
    }}
  ],
  "strengths": [""],
  "recommendation": ""
}}

Rules:
- overall = round(completeness×0.25 + accuracy×0.25 + cross_section_consistency×0.20 + entity_categorisation×0.15 + retrieval_utility×0.15)
- Empty fields (None) are acceptable when the product type makes them inapplicable
- issues may be []
- strengths must have at least one entry\
"""


In [6]:
# ── CELL 6 — Judge function ───────────────────────────────────────────────────
def judge_document(entity, document, doc_type='structured'):
    prompt_template = STRUCTURED_JUDGE if doc_type == 'structured' else FULL_JUDGE

    prompt = prompt_template.format(
        entity=json.dumps(entity, ensure_ascii=False, indent=2),
        document=document
    )

    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt}
            ],
            response_format={'type': 'json_object'},
            temperature=0,
            max_completion_tokens=1024,
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f'  ERROR {entity["product_id"]}: {e}')
        return None

In [7]:
# ── CELL 7 — Test judge on 3 products before full run ────────────────────────
# Test on: microbial (AF0001), blank ingredients (AF0014), pesticide (PN0001)
print('=== STRUCTURED JUDGE TEST ===')
for pid in ['AF0001', 'AF0014', 'PN0001']:
    entity   = entities[pid]
    document = structured_docs[pid]['document']
    result   = judge_document(entity, document, doc_type='structured')
    if result:
        print(f'\n{pid} | {entity["product_name"]}')
        print(f'  Scores: {result["scores"]}')
        print(f'  Issues: {len(result.get("issues",[]))}')
        if result.get('issues'):
            for iss in result['issues'][:2]:
                print(f'    [{iss["severity"]}] {iss["type"]}: {iss["description"][:80]}')
        print(f'  Recommendation: {result.get("recommendation","")[:100]}')

=== STRUCTURED JUDGE TEST ===

AF0001 | Citrus Junliqing
  Scores: {'completeness': 10, 'accuracy': 10, 'entity_categorisation': 10, 'retrieval_utility': 10, 'overall': 10}
  Issues: 0
  Recommendation: No changes needed; the retrieval document faithfully represents the source.

AF0014 | Chinese Cabbage and Cabbage Water-Soluble Fertilizer
  Scores: {'completeness': 10, 'accuracy': 10, 'entity_categorisation': 10, 'retrieval_utility': 10, 'overall': 10}
  Issues: 0
  Recommendation: No changes needed; the retrieval document faithfully represents the structured entity source.

PN0001 | Bacillus thuringiensis 4000 IU/μL
  Scores: {'completeness': 10, 'accuracy': 10, 'entity_categorisation': 10, 'retrieval_utility': 10, 'overall': 10}
  Issues: 0
  Recommendation: No changes needed; the retrieval document is a faithful and complete representation of the structure


In [8]:
# ── CELL 8 — Review test output before full run ───────────────────────────────
# Check:
# 1. AF0014 — blank ingredients should NOT be penalised (honest null)
# 2. Scores should be mostly 7-9 range
# 3. Issues should be specific and actionable, not generic
# Only proceed to Cell 9 if test results look reasonable
print('\nReview scores above.')
print('AF0014 blank ingredients should score 8+ on completeness (honest null).')
print('If scores look reasonable, proceed to Cell 9.')


Review scores above.
AF0014 blank ingredients should score 8+ on completeness (honest null).
If scores look reasonable, proceed to Cell 9.


In [9]:
# ── CELL 9 — Full structured judge run ───────────────────────────────────────
structured_results = []
failed_s = []

for pid, entity in tqdm(entities.items(), desc='Judging structured docs'):
    document = structured_docs[pid]['document']
    result   = judge_document(entity, document, doc_type='structured')
    if result:
        structured_results.append(result)
    else:
        failed_s.append(pid)

print(f'\nJudged: {len(structured_results)} / {len(entities)}')
if failed_s:
    print(f'Failed: {failed_s}')

# Score distribution
scores = [r['scores']['overall'] for r in structured_results]
print(f'\nOverall scores:')
print(f'  Min:     {min(scores):.1f}')
print(f'  Max:     {max(scores):.1f}')
print(f'  Average: {sum(scores)/len(scores):.1f}')
print(f'  ≥ 7:     {sum(1 for s in scores if s >= 7)} products')
print(f'  5-6:     {sum(1 for s in scores if 5 <= s < 7)} products')
print(f'  < 5:     {sum(1 for s in scores if s < 5)} products — REVIEW NEEDED')

Judging structured docs: 100%|██████████| 114/114 [06:32<00:00,  3.44s/it]


Judged: 114 / 114

Overall scores:
  Min:     10.0
  Max:     10.0
  Average: 10.0
  ≥ 7:     114 products
  5-6:     0 products
  < 5:     0 products — REVIEW NEEDED


In [16]:
# ── Re-run full judge with corrected prompt ───────────────────────────────────
full_results = []
failed_f     = []

for pid, entity in tqdm(entities.items(), desc='Judging full docs (corrected)'):
    document = full_docs[pid]['document']
    result   = judge_document_retry(entity, document, doc_type='full')
    if result:
        full_results.append(result)
    else:
        failed_f.append(pid)

print(f'\nJudged: {len(full_results)} / {len(entities)}')
if failed_f:
    print(f'Failed: {failed_f}')

scores_f = [r['scores']['overall'] for r in full_results]
print(f'\nOverall scores:')
print(f'  Min:     {min(scores_f):.1f}')
print(f'  Max:     {max(scores_f):.1f}')
print(f'  Average: {sum(scores_f)/len(scores_f):.1f}')
print(f'  ≥ 7:     {sum(1 for s in scores_f if s >= 7)} products')
print(f'  5-6:     {sum(1 for s in scores_f if 5 <= s < 7)} products')
print(f'  < 5:     {sum(1 for s in scores_f if s < 5)} products')

Judging full docs (corrected): 100%|██████████| 114/114 [13:19<00:00,  7.02s/it]


Judged: 114 / 114

Overall scores:
  Min:     7.0
  Max:     10.0
  Average: 9.0
  ≥ 7:     114 products
  5-6:     0 products
  < 5:     0 products


In [11]:
# Retry the 3 failed products with higher token limit
def judge_document_retry(entity, document, doc_type='full'):
    prompt_template = FULL_JUDGE
    prompt = prompt_template.format(
        entity=json.dumps(entity, ensure_ascii=False, indent=2),
        document=document
    )
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': prompt}
            ],
            response_format={'type': 'json_object'},
            temperature=0,
            max_completion_tokens=2048,  # doubled
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f'  ERROR {entity["product_id"]}: {e}')
        return None

for pid in ['AF0020', 'AF0022', 'PN0035']:
    entity   = entities[pid]
    document = full_docs[pid]['document']
    result   = judge_document_retry(entity, document, doc_type='full')
    if result:
        full_results.append(result)
        print(f'{pid}: score={result["scores"]["overall"]} issues={len(result.get("issues",[]))}')
    else:
        print(f'{pid}: STILL FAILING')

AF0020: score=8 issues=10
AF0022: score=8 issues=11
PN0035: score=7 issues=15


In [17]:
# ── CELL 11 — Analyse results ────────────────────────────────────────────────
print('=== FULL — PRODUCTS NEEDING ATTENTION (score < 8) ===')
low_f = [r for r in full_results if r['scores']['overall'] < 8]
if low_f:
    for r in sorted(low_f, key=lambda x: x['scores']['overall']):
        print(f"  {r['product_id']} | score={r['scores']['overall']} | {r['recommendation'][:100]}")
else:
    print('  None ✓')

print('\n=== ISSUE TYPES (full) ===')
from collections import Counter
issue_types_f = Counter(
    i['type'] for r in full_results for i in r.get('issues', [])
)
for issue_type, count in issue_types_f.most_common():
    print(f'  {count:>3}x  {issue_type}')

print(f'\nTotal issues flagged: {sum(issue_types_f.values())}')
print(f'Products with zero issues: {sum(1 for r in full_results if not r.get("issues"))}')

=== FULL — PRODUCTS NEEDING ATTENTION (score < 8) ===
  PN0004 | score=7 | Keep the structured summary as-is, but correct or remove the contradictory supplementary label text 
  PN0005 | score=7 | Correct the active ingredient and chemical-class inconsistency across sections as the highest priori
  PN0007 | score=7 | Retain the structured summary, but urgently review and correct both prose sections for active ingred
  PN0008 | score=7 | Correct the active ingredient identities and percentages across all sections first, then harmonise t
  PN0011 | score=7 | Correct the ORIGINAL PRODUCT DESCRIPTION to use Lambda-cyhalothrin consistently instead of Beta-cyfl
  PN0014 | score=7 | Correct the core chemistry across sections so product name and active ingredients align consistently
  PN0031 | score=7 | Correct the active ingredient translation and normalization across all sections so that the English 
  PN0037 | score=7 | Correct the English product name or the active-ingredient fields so tha

In [13]:
# Show sample issues from full results to understand what's being flagged
print('=== SAMPLE HALLUCINATION ISSUES ===')
hall_issues = [
    (r['product_id'], i)
    for r in full_results
    for i in r.get('issues', [])
    if i['type'] == 'HALLUCINATION'
][:10]
for pid, issue in hall_issues:
    print(f'  {pid} [{issue["severity"]}] {issue["section"]}: {issue["description"][:100]}')

print('\n=== SAMPLE CROSS_SECTION_CONFLICT ISSUES ===')
cross_issues = [
    (r['product_id'], i)
    for r in full_results
    for i in r.get('issues', [])
    if i['type'] == 'CROSS_SECTION_CONFLICT'
][:10]
for pid, issue in cross_issues:
    print(f'  {pid} [{issue["severity"]}] {issue["section"]}: {issue["description"][:100]}')

=== SAMPLE HALLUCINATION ISSUES ===
  AF0001 [medium] ORIGINAL PRODUCT DESCRIPTION: The prose adds unsupported claims not present in the ground truth, including inhibition of overwinte
  AF0001 [medium] 中文产品说明: The Chinese prose adds unsupported claims beyond the source, including suppression of 越冬害虫和虫卵, formi
  AF0002 [medium] ORIGINAL PRODUCT DESCRIPTION: The prose adds claims about inhibition of overwintering pests and pest eggs, frost/cold protection, 
  AF0002 [medium] 中文产品说明: The Chinese prose adds claims about suppressing 越冬害虫和虫卵, 清园剂用途, and protection for safe overwinterin
  AF0003 [medium] ORIGINAL PRODUCT DESCRIPTION: The prose adds claims not present in the structured source, including inhibition of overwintering pe
  AF0003 [medium] 中文产品说明: 中文说明增加了结构化来源中没有的内容，如‘清园剂’、对越冬害虫和虫卵的抑制、抗冻害霜害、帮助安全过冬，以及生产销售信息等。
  AF0004 [medium] ORIGINAL PRODUCT DESCRIPTION: The prose adds efficacy claims not present in the structured source, including reducing risks from c
  AF0005 [medium] ORIGINAL

In [15]:
# Re-test 5 products with updated prompt
print('=== RE-TEST WITH UPDATED PROMPT ===')
for pid in ['AF0001', 'AF0002', 'PN0004', 'PN0007', 'PN0037']:
    entity   = entities[pid]
    document = full_docs[pid]['document']
    result   = judge_document_retry(entity, document, doc_type='full')
    if result:
        print(f'{pid}: overall={result["scores"]["overall"]} '
              f'issues={len(result.get("issues",[]))} '
              f'hallucinations={sum(1 for i in result.get("issues",[]) if i["type"]=="HALLUCINATION")}')

=== RE-TEST WITH UPDATED PROMPT ===
AF0001: overall=8 issues=3 hallucinations=0
AF0002: overall=8 issues=4 hallucinations=0
PN0004: overall=7 issues=5 hallucinations=2
PN0007: overall=7 issues=5 hallucinations=2
PN0037: overall=7 issues=5 hallucinations=2


In [18]:
# ── CELL 12 — Save all results ────────────────────────────────────────────────
with open('judge_results_structured_v2.json', 'w', encoding='utf-8') as f:
    json.dump(structured_results, f, ensure_ascii=False, indent=2)
print('Saved → judge_results_structured_v2.json')

with open('judge_results_full_v2.json', 'w', encoding='utf-8') as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)
print('Saved → judge_results_full_v2.json')

scores_s = [r['scores']['overall'] for r in structured_results]
scores_f = [r['scores']['overall'] for r in full_results]

summary = {
    'structured': {
        'total':  len(structured_results),
        'avg':    round(sum(scores_s)/len(scores_s), 2),
        'min':    min(scores_s),
        'max':    max(scores_s),
        'pass_7': sum(1 for s in scores_s if s >= 7),
        'fail':   sum(1 for s in scores_s if s < 7),
    },
    'full': {
        'total':  len(full_results),
        'avg':    round(sum(scores_f)/len(scores_f), 2),
        'min':    min(scores_f),
        'max':    max(scores_f),
        'pass_7': sum(1 for s in scores_f if s >= 7),
        'fail':   sum(1 for s in scores_f if s < 7),
        'products_score_7': [r['product_id'] for r in full_results
                             if r['scores']['overall'] == 7],
        'source_data_issues': {
            r['product_id']: r['recommendation']
            for r in full_results if r['scores']['overall'] < 8
        }
    }
}

with open('judge_summary_v2.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved → judge_summary_v2.json')

from google.colab import files
files.download('judge_results_structured_v2.json')
files.download('judge_results_full_v2.json')
files.download('judge_summary_v2.json')

print(f"""
=== JUDGE COMPLETE ✓ ===
Structured: avg={summary['structured']['avg']}  pass={summary['structured']['pass_7']}/114
Full:       avg={summary['full']['avg']}  pass={summary['full']['pass_7']}/114
All 114 documents cleared for embedding.
9 PN products flagged for client data review (score=7, source conflicts).
Next: notebook_06_embedding.ipynb
""")

Saved → judge_results_structured_v2.json
Saved → judge_results_full_v2.json
Saved → judge_summary_v2.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


=== JUDGE COMPLETE ✓ ===
Structured: avg=10.0  pass=114/114
Full:       avg=8.97  pass=114/114
All 114 documents cleared for embedding.
9 PN products flagged for client data review (score=7, source conflicts).
Next: notebook_06_embedding.ipynb

